![VGG.png](./images/VGG-16.png)

In [1]:
import torch
import torch.nn as nn

In [2]:
def VGG_block(num_convs,num_in_channels,num_out_channels):
    layers = []
    for _ in range(num_convs):
        layers.append(nn.Conv2d(in_channels=num_in_channels,out_channels=num_out_channels,kernel_size=3,padding=1))
        layers.append(nn.ReLU())
        num_in_channels = num_out_channels
    layers.append(nn.MaxPool2d(kernel_size=2,stride=2))
    return nn.Sequential(*layers)

In [3]:
class VGG(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv_arch = [(1,64),(1,128),(2,256),(2,256),(2,512)]
        self.vgg_blks = []
        self.num_in_channels = 1
        for num_convs,self.num_out_channels in self.conv_arch:
            self.vgg_blks.append(VGG_block(num_convs,self.num_in_channels,self.num_out_channels))
            self.num_in_channels = self.num_out_channels
        self.net = nn.Sequential(*self.vgg_blks,
                                 nn.Flatten(),
                                 nn.Linear(self.num_out_channels * 7 * 7,4096),nn.ReLU(),nn.Dropout(p=0.5),
                                 nn.Linear(4096,4096),nn.ReLU(),nn.Dropout(p=0.5),
                                nn.Linear(4096,10))
    def forward(self,X):
        return self.net(X)

In [4]:
X = torch.randn(1,1,224,224)
vgg = VGG()
for layer in vgg.net:
    X = layer(X)
    print(layer.__class__.__name__,'output shape:\t',X.shape)

Sequential output shape:	 torch.Size([1, 64, 112, 112])
Sequential output shape:	 torch.Size([1, 128, 56, 56])
Sequential output shape:	 torch.Size([1, 256, 28, 28])
Sequential output shape:	 torch.Size([1, 256, 14, 14])
Sequential output shape:	 torch.Size([1, 512, 7, 7])
Flatten output shape:	 torch.Size([1, 25088])
Linear output shape:	 torch.Size([1, 4096])
ReLU output shape:	 torch.Size([1, 4096])
Dropout output shape:	 torch.Size([1, 4096])
Linear output shape:	 torch.Size([1, 4096])
ReLU output shape:	 torch.Size([1, 4096])
Dropout output shape:	 torch.Size([1, 4096])
Linear output shape:	 torch.Size([1, 10])


In [ ]:
from py09_FashionMNIST_Dataset import load_data_fashion_mnist
from deeplearn_tools import Animator,Timer,evaluate_accuracy

In [ ]:
num_epoches = 10
batch_size = 32
lr = 0.01
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(vgg.parameters(),lr=lr)
train_loader,test_loader = load_data_fashion_mnist(batch_size,resize=224)

In [ ]:
def train(train_loader,test_loader,net,num_epoches,loss_fn,optimizer,device="cuda"):
    def init_weights(layer):
        if type(layer)== nn.Linear or type(layer) == nn.Conv2d:
            nn.init.xavier_uniform_(layer.weight)
    net.apply(init_weights)
    net.to(device)
    data_dis = Animator(xlabel='num_epoch',xlim=[1,num_epoches],legend=['train_loss','train_accuracy','test_accuracy'],yscale='log')
    timer = Timer()
    for epoch in range(num_epoches):
        l_sum = 0
        timer.start()
        for X,y in train_loader:
            X,y = X.to(device),y.to(device)
            optimizer.zero_grad()
            y_hat = net(X)
            l = loss_fn(y_hat,y)
            l.backward()
            optimizer.step()
            l_sum += l
        epoch_time = timer.stop()
        train_acc = evaluate_accuracy(net,train_loader,device)
        test_acc = evaluate_accuracy(net,test_loader,device)
        print(f"{epoch}:train loss,{l_sum:.2f},train_acc:{train_acc},test_acc:{test_acc},time:{epoch_time:.4f}s")
        data_dis.add(epoch+1,[l_sum,train_acc,test_acc])

In [ ]:
train(train_loader,test_loader,vgg,num_epoches,loss_fn,optimizer,device="cuda")